In [5]:
import openpyxl
import pandas as pd

# Load the workbook
wb = openpyxl.load_workbook('PRISMA_Consolidated.xlsx')
ws = wb['Consolidated DataFrame']

# Read data into a list of dictionaries, SKIP EMPTY ROWS
headers = []
for col in range(1, ws.max_column + 1):
    headers.append(ws.cell(1, col).value)

data = []
for row in range(2, ws.max_row + 1):
    record = {}
    is_empty = True
    for col_idx, header in enumerate(headers, start=1):
        value = ws.cell(row, col_idx).value
        record[header] = value
        if value is not None:  # Check if row has any data
            is_empty = False
    
    if not is_empty:  # Only add non-empty rows
        data.append(record)

print(f"Total records before deduplication: {len(data)}")

# Convert to DataFrame
df = pd.DataFrame(data)

# Separate records with and without DOI
df_with_doi = df[df['DOI'].notna()].copy()
df_without_doi = df[df['DOI'].isna()].copy()

print(f"Records with DOI: {len(df_with_doi)}")
print(f"Records without DOI: {len(df_without_doi)}")

# Remove duplicates based on DOI (keep first occurrence)
df_with_doi_dedup = df_with_doi.drop_duplicates(subset=['DOI'], keep='first')

duplicates_removed = len(df_with_doi) - len(df_with_doi_dedup)
print(f"Duplicates removed: {duplicates_removed}")

# Combine back
df_final = pd.concat([df_with_doi_dedup, df_without_doi], ignore_index=True)

print(f"Total records after deduplication: {len(df_final)}")

# Save deduplicated records to Duplicate Review sheet
ws_review = wb['Duplicate Review']

# Clear existing content in Duplicate Review sheet
for row in ws_review.iter_rows():
    for cell in row:
        cell.value = None

# Write headers to Duplicate Review sheet
for col_idx, header in enumerate(headers, start=1):
    ws_review.cell(1, col_idx, header)
    ws_review.cell(1, col_idx).font = openpyxl.styles.Font(bold=True)

# Write deduplicated data to Duplicate Review sheet
for row_idx, record in enumerate(df_final.to_dict('records'), start=2):
    for col_idx, header in enumerate(headers, start=1):
        ws_review.cell(row_idx, col_idx, record[header])

# Auto-adjust column widths for Duplicate Review sheet
from openpyxl.utils import get_column_letter
for col_idx, header in enumerate(headers, start=1):
    column_letter = get_column_letter(col_idx)
    if header == 'Title':
        ws_review.column_dimensions[column_letter].width = 50
    elif header == 'Author/(s)':
        ws_review.column_dimensions[column_letter].width = 30
    elif header == 'Abstract':
        ws_review.column_dimensions[column_letter].width = 60
    elif header == 'Keywords':
        ws_review.column_dimensions[column_letter].width = 30
    elif header == 'Year':
        ws_review.column_dimensions[column_letter].width = 10
    elif header == 'Platform':
        ws_review.column_dimensions[column_letter].width = 20
    elif header == 'DOI':
        ws_review.column_dimensions[column_letter].width = 25

# Save the workbook
wb.save('PRISMA_Deduplicated.xlsx')

print(f"\nDeduplication complete!")
print(f"Deduplicated records ({len(df_final)}) saved to 'Duplicate Review' sheet")

Total records before deduplication: 7894
Records with DOI: 5542
Records without DOI: 2352
Duplicates removed: 1544
Total records after deduplication: 6350

Deduplication complete!
Deduplicated records (6350) saved to 'Duplicate Review' sheet
